In [2]:
!unzip -o /content/restaurant_data_package.zip -d /content/

Archive:  /content/restaurant_data_package.zip
  inflating: /content/dataset/restaurant_test_30k.json  
  inflating: /content/dataset/restaurant_train_70k.json  
  inflating: /content/dataset/synthetic_reviews_multi_style_clipped.csv  
  inflating: /content/dataset/embeddings_base_70k.npy  
  inflating: /content/dataset/embeddings_synthetic_multi_style_clipped.npy  


In [3]:
!ls /content/dataset

embeddings_base_70k.npy
embeddings_synthetic_multi_style_clipped.npy
restaurant_test_30k.json
restaurant_train_70k.json
synthetic_reviews_multi_style_clipped.csv


In [4]:
!pip install transformers datasets scikit-learn -q

In [7]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict

# ── CONFIG ──────────────────────────────────────────────
GUIDE_SIZE   = 3000   # how many real train samples to use as tg_guide
SYNTH_SIZE   = 30000  # how many synthetic samples to use (should be ~10x guide)
RANDOM_STATE = 42
SAMPLES_PER_STAR_GUIDE = GUIDE_SIZE // 5   # 600 per star
SAMPLES_PER_STAR_SYNTH = SYNTH_SIZE // 5   # 6000 per star
# ────────────────────────────────────────────────────────

# --- ts: synthetic reviews + precomputed embeddings ---
synth_df = pd.read_csv('/content/dataset/synthetic_reviews_multi_style_clipped.csv')
synth_df = synth_df.rename(columns={'full_text': 'text', 'stars': 'label'})
synth_df['label'] = synth_df['label'].astype(float)

synth_embs = np.load('/content/dataset/embeddings_synthetic_multi_style_clipped.npy')
assert len(synth_embs) == len(synth_df), "Embedding count doesn't match synthetic df"
synth_df['emb_idx'] = np.arange(len(synth_df))  # track original index for embedding lookup

# Stratified sample of synthetic data
synth_sampled = (
    synth_df.groupby('label', group_keys=False)
            .apply(lambda x: x.sample(
                n=min(SAMPLES_PER_STAR_SYNTH, len(x)),
                random_state=RANDOM_STATE
            ))
            .reset_index(drop=True)
)
synth_embs_sampled = synth_embs[synth_sampled['emb_idx'].values]
synth_sampled = synth_sampled.drop(columns=['emb_idx'])

print(f"Synthetic (ts) distribution:\n{synth_sampled['label'].value_counts().sort_index().to_string()}")

# --- tg_guide: real train reviews + precomputed embeddings (stratified subset) ---
guide_full_df = pd.read_json('/content/dataset/restaurant_train_70k.json', lines=True)
guide_full_df = guide_full_df.rename(columns={'stars': 'label'})
guide_full_df['label'] = guide_full_df['label'].astype(float)

guide_full_embs = np.load('/content/dataset/embeddings_base_70k.npy')
assert len(guide_full_embs) == len(guide_full_df), "Embedding count doesn't match guide df"
guide_full_df['emb_idx'] = np.arange(len(guide_full_df))

# Stratified sample down to GUIDE_SIZE
guide_df = (
    guide_full_df.groupby('label', group_keys=False)
                 .apply(lambda x: x.sample(
                     n=min(SAMPLES_PER_STAR_GUIDE, len(x)),
                     random_state=RANDOM_STATE
                 ))
                 .reset_index(drop=True)
)
guide_embs = guide_full_embs[guide_df['emb_idx'].values]
guide_df = guide_df.drop(columns=['emb_idx'])

print(f"\nGuide (tg_guide) distribution:\n{guide_df['label'].value_counts().sort_index().to_string()}")

# --- tg_eval: real test reviews (locked) ---
eval_df = pd.read_json('/content/dataset/restaurant_test_30k.json', lines=True)
eval_df = eval_df.rename(columns={'stars': 'label'})
eval_df['label'] = eval_df['label'].astype(float)

print(f"\ntg_eval distribution:\n{eval_df['label'].value_counts().sort_index().to_string()}")

# --- ts train/val split (both synthetic) ---
train_df = synth_sampled.sample(frac=0.7, random_state=RANDOM_STATE)
val_df   = synth_sampled.drop(train_df.index)

# Keep track of which synthetic embeddings go to train vs val
train_embs = synth_embs_sampled[train_df.index.values]
val_embs   = synth_embs_sampled[val_df.index.values]

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# --- HuggingFace DatasetDict (text + label only, embeddings stored as numpy separately) ---
ds = DatasetDict({
    'train':      Dataset.from_pandas(train_df[['text', 'label']]),
    'validation': Dataset.from_pandas(val_df[['text', 'label']]),
    'guide':      Dataset.from_pandas(guide_df[['text', 'label']]),
    'test':       Dataset.from_pandas(eval_df[['text', 'label']].reset_index(drop=True))
})

print(f"\n{'Split':<14}{'Size':<10}")
print("-" * 24)
for split in ['train', 'validation', 'guide', 'test']:
    print(f"{split:<14}{len(ds[split]):<10}")

print(f"\nEmbedding shapes:")
print(f"  train_embs:  {train_embs.shape}")
print(f"  val_embs:    {val_embs.shape}")
print(f"  guide_embs:  {guide_embs.shape}")

/tmp/ipykernel_3991/3933469581.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


Synthetic (ts) distribution:
label
1.0    5950
2.0    5949
3.0    5980
4.0    5960
5.0    5952


/tmp/ipykernel_3991/3933469581.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(



Guide (tg_guide) distribution:
label
1.0    600
2.0    600
3.0    600
4.0    600
5.0    600

tg_eval distribution:
label
1.0    6000
2.0    6000
3.0    6000
4.0    6000
5.0    6000

Split         Size      
------------------------
train         20854     
validation    8937      
guide         3000      
test          30000     

Embedding shapes:
  train_embs:  (20854, 384)
  val_embs:    (8937, 384)
  guide_embs:  (3000, 384)


In [8]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from datasets import Dataset, DatasetDict
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm

# Config — change these as needed
MODEL_NAME = "microsoft/deberta-v3-small"  # smaller than base, fits on Colab
MAX_LEN = 256
BATCH_SIZE = 64
NUM_EPOCHS = 5
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Auto-detect best supported dtype
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 8:          # Ampere and above (A100, A10, RTX 3090+)
        DTYPE = torch.bfloat16
        BATCH_SIZE = 64
    else:                     # V100, T4, P100 etc — bf16 not supported
        DTYPE = torch.float32
        BATCH_SIZE = 64
else:
    DTYPE = torch.float32     # CPU — always fp32

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Compute capability: {cap if torch.cuda.is_available() else 'N/A'}")
print(f"Using dtype: {DTYPE}")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Compute capability: (8, 0)
Using dtype: torch.bfloat16


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

ds_tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization done.")
print(ds_tokenized)

Map:   0%|          | 0/20854 [00:00<?, ? examples/s]

Map:   0%|          | 0/8937 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Tokenization done.
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20854
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8937
    })
    guide: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30000
    })
})


In [10]:
def compute_qwk(preds, labels):
    # Clip predictions to valid range and round to nearest integer for QWK
    preds_rounded = np.clip(np.round(preds), 1, 5).astype(int)
    labels_int = np.array(labels).astype(int)
    return cohen_kappa_score(labels_int, preds_rounded, weights="quadratic")

# Quick sanity check
print("QWK sanity check:", compute_qwk([1,2,3,4,5], [1,2,3,4,5]))  # should be 1.0

QWK sanity check: 1.0


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True,
    torch_dtype=DTYPE
)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

sample_weights = np.ones(len(ds_tokenized["train"]))

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(ds_tokenized["train"],      batch_size=BATCH_SIZE, sampler=train_sampler)
val_loader   = DataLoader(ds_tokenized["validation"], batch_size=BATCH_SIZE, shuffle=False)
guide_loader = DataLoader(ds_tokenized["guide"],      batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(ds_tokenized["test"],       batch_size=BATCH_SIZE, shuffle=False)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

loss_fn = nn.MSELoss()
print(f"Model dtype:      {next(model.parameters()).dtype}")
print(f"Steps per epoch:  {len(train_loader)}")
print(f"Total steps:      {total_steps}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Model dtype:      torch.bfloat16
Steps per epoch:  326
Total steps:      1630


In [12]:
def evaluate(loader, split_name="val"):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].float().to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.squeeze(-1)

            # cast to float32 before moving to cpu — fixes bf16/fp16 numpy error
            all_preds.extend(preds.float().cpu().numpy())
            all_labels.extend(labels.float().cpu().numpy())

    qwk = compute_qwk(all_preds, all_labels)
    mse = np.mean((np.array(all_preds) - np.array(all_labels)) ** 2)
    print(f"  [{split_name}] QWK: {qwk:.4f} | MSE: {mse:.4f}")
    return qwk, all_preds, all_labels

In [13]:
import os, json
import numpy as np

SAVE_DIR = "/content/baseline_model"
os.makedirs(SAVE_DIR, exist_ok=True)

history = {
    "train_loss": [],
    "synth_val_qwk": [],
    "real_guide_qwk": [],   # guide set performance — this becomes reweighting signal later
    "real_test_qwk": []
}

best_test_qwk = -1
best_epoch = -1

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in loop:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].float().to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        preds  = outputs.logits.squeeze(-1).to(DTYPE)
        labels = labels.to(DTYPE)

        loss = loss_fn(preds, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | Avg Train Loss: {avg_loss:.4f}")
    synth_val_qwk,  _, _ = evaluate(val_loader,   "synth_val ")
    real_guide_qwk, _, _ = evaluate(guide_loader, "real_guide")
    real_test_qwk,  _, _ = evaluate(test_loader,  "real_test ")

    history["train_loss"].append(avg_loss)
    history["synth_val_qwk"].append(synth_val_qwk)
    history["real_guide_qwk"].append(real_guide_qwk)
    history["real_test_qwk"].append(real_test_qwk)

    # Save best model by real_test_qwk
    if real_test_qwk > best_test_qwk:
        best_test_qwk = real_test_qwk
        best_epoch = epoch + 1
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✓ Best model saved (test QWK: {best_test_qwk:.4f})")

# Save history
with open(f"{SAVE_DIR}/history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\n=== BASELINE COMPLETE ===")
print(f"Best real_test QWK: {best_test_qwk:.4f} at epoch {best_epoch}")
print(f"\nFull history:")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Synth Val':<14}{'Real Guide':<14}{'Real Test':<14}")
print("-" * 60)
for i in range(NUM_EPOCHS):
    print(f"{i+1:<8}{history['train_loss'][i]:<14.4f}{history['synth_val_qwk'][i]:<14.4f}"
          f"{history['real_guide_qwk'][i]:<14.4f}{history['real_test_qwk'][i]:<14.4f}")

Epoch 1/5:   0%|          | 0/326 [00:00<?, ?it/s]


Epoch 1/5 | Avg Train Loss: 4.5570
  [synth_val ] QWK: 0.8800 | MSE: 0.4169
  [real_guide] QWK: 0.7800 | MSE: 0.6590
  [real_test ] QWK: 0.7848 | MSE: 0.6618


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (test QWK: 0.7848)


Epoch 2/5:   0%|          | 0/326 [00:00<?, ?it/s]

...


Epoch 4/5 | Avg Train Loss: 0.2384
  [synth_val ] QWK: 0.8993 | MSE: 0.3667
  [real_guide] QWK: 0.8071 | MSE: 0.5998
  [real_test ] QWK: 0.8051 | MSE: 0.6029


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (test QWK: 0.8051)


Epoch 5/5:   0%|          | 0/326 [00:00<?, ?it/s]


Epoch 5/5 | Avg Train Loss: 0.2402
  [synth_val ] QWK: 0.8995 | MSE: 0.3659
  [real_guide] QWK: 0.8078 | MSE: 0.5997
  [real_test ] QWK: 0.8049 | MSE: 0.6025

=== BASELINE COMPLETE ===
Best real_test QWK: 0.8051 at epoch 4

Full history:
Epoch   Train Loss    Synth Val     Real Guide    Real Test     
------------------------------------------------------------
1       4.5570        0.8800        0.7800        0.7848        
2       0.2843        0.8819        0.7995        0.7959        
3       0.2499        0.8939        0.8076        0.8038        
4       0.2384        0.8993        0.8071        0.8051        
5       0.2402        0.8995        0.8078        0.8049        


In [16]:
!zip /content/baseline_model.zip -r /content/baseline_model

  adding: content/baseline_model/ (stored 0%)
  adding: content/baseline_model/model.safetensors (deflated 21%)
  adding: content/baseline_model/config.json (deflated 55%)
  adding: content/baseline_model/tokenizer_config.json (deflated 48%)
  adding: content/baseline_model/tokenizer.json (deflated 79%)
  adding: content/baseline_model/history.json (deflated 52%)
